### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [14]:
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
import os

In [5]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: Embeddings.pdf
  ✓ Loaded 8 pages

Processing: Retrieval-Augmented_Generation_RAG.pdf
  ✓ Loaded 12 pages

Total documents loaded: 20


In [8]:
### Split the documents into smaller chunks
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks"""
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap,separators=["\n\n", "\n", " ", ""])
    all_chunks = []
    
    for doc in documents:
        try:
            chunks = text_splitter.split_text(doc.page_content)
            for i, chunk in enumerate(chunks):
                chunk_doc = Document(
                    page_content=chunk,
                    metadata={
                        **doc.metadata,
                        'chunk_index': i,
                        'original_length': len(doc.page_content),
                        'chunk_length': len(chunk)
                    }
                )
                all_chunks.append(chunk_doc)
            print(f"  ✓ Split document into {len(chunks)} chunks")
        except Exception as e:
            print(f"  ✗ Error splitting document: {e}")
    
    print(f"\nTotal chunks created: {len(all_chunks)}")
    return all_chunks
# Split the loaded PDF documents into chunks
chunked_documents = split_documents(all_pdf_documents)
chunked_documents


  ✓ Split document into 3 chunks
  ✓ Split document into 6 chunks
  ✓ Split document into 5 chunks
  ✓ Split document into 6 chunks
  ✓ Split document into 5 chunks
  ✓ Split document into 5 chunks
  ✓ Split document into 5 chunks
  ✓ Split document into 2 chunks
  ✓ Split document into 5 chunks
  ✓ Split document into 7 chunks
  ✓ Split document into 5 chunks
  ✓ Split document into 5 chunks
  ✓ Split document into 6 chunks
  ✓ Split document into 7 chunks
  ✓ Split document into 7 chunks
  ✓ Split document into 5 chunks
  ✓ Split document into 9 chunks
  ✓ Split document into 9 chunks
  ✓ Split document into 8 chunks
  ✓ Split document into 6 chunks

Total chunks created: 116


[Document(metadata={'producer': 'Adobe PDF Library 24.2.255', 'creator': 'Acrobat PDFMaker 24 für Word', 'creationdate': '2024-08-29T13:20:26+02:00', 'author': 'carsten caspary', 'company': '', 'grammarlydocumentid': '3f9ca6d320641fb196deb53c5584fe5bf063c26b10bbb6d2b96725cb7e9f443a', 'keywords': '', 'msip_label_1b52b3a1-dbcb-41fb-a452-370cf542753f_actionid': 'e39e1e20-c3e4-466a-add7-e012621483d9', 'msip_label_1b52b3a1-dbcb-41fb-a452-370cf542753f_contentbits': '0', 'msip_label_1b52b3a1-dbcb-41fb-a452-370cf542753f_enabled': 'true', 'msip_label_1b52b3a1-dbcb-41fb-a452-370cf542753f_method': 'Privileged', 'msip_label_1b52b3a1-dbcb-41fb-a452-370cf542753f_name': 'Public', 'msip_label_1b52b3a1-dbcb-41fb-a452-370cf542753f_setdate': '2024-04-15T05:12:54Z', 'msip_label_1b52b3a1-dbcb-41fb-a452-370cf542753f_siteid': 'd1323671-cdbe-4417-b4d4-bdb24b51316b', 'moddate': '2024-08-29T13:20:35+02:00', 'sourcemodified': '', 'subject': '', 'title': '', 'source': '..\\data\\pdf\\Embeddings.pdf', 'total_pages

In [11]:
### Embedding and storing in vector database

import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\Computer Care\RAG-learning\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [26]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2646.35it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


### Saving to vector db(funcionalities)



In [15]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [17]:
### generating text from chunks 
texts=[doc.page_content for doc in chunked_documents]
### generate embeddings
embeddings=embedding_manager.generate_embeddings(texts)
### add to vector store
vectorstore.add_documents(chunked_documents,embeddings)

Generating embeddings for 116 texts...


Batches: 100%|██████████| 4/4 [00:12<00:00,  3.07s/it]


Generated embeddings with shape: (116, 384)
Adding 116 documents to vector store...
Successfully added 116 documents to vector store
Total documents in collection: 116


### Retriever Pipeline From VectorStore

In [28]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [29]:
rag_retriever

In [33]:
rag_retriever.retrieve("What is Retrieval-Augmented Generation (RAG)?")

Retrieving documents for query: 'What is Retrieval-Augmented Generation (RAG)?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 30.98it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_73ccd066_92',
  'content': 'doi.org/10.1162/coli_a_00524\n123\nM. Klesel, H. F. Wittmann: Retrieval-Augmented Generation (RAG), Bus Inf Syst Eng 67(4):551–561 (2025) 559\nContent courtesy of Springer Nature, terms of use apply. Rights reserved.',
  'metadata': {'original_length': 6565,
   'content_length': 213,
   'file_type': 'pdf',
   'chunk_length': 213,
   'creationdate': '2025-09-18T03:03:54+02:00',
   'moddate': '2025-09-18T03:03:54+02:00',
   'producer': 'iText® 5.5.13.3 ©2000-2022 iText Group NV (SPRINGER SBM; licensed version)',
   'source_file': 'Retrieval-Augmented_Generation_RAG.pdf',
   'total_pages': 12,
   'chunk_index': 8,
   'page': 8,
   'source': '..\\data\\pdf\\Retrieval-Augmented_Generation_RAG.pdf',
   'page_label': '9',
   'creator': 'PyPDF',
   'doc_index': 92},
  'similarity_score': 0.513978123664856,
  'distance': 0.48602187633514404,
  'rank': 1},
 {'id': 'doc_0ece0cad_83',
  'content': 'issue.\nRetrieval effectiveness The effectiveness of a RAG 

### RAG Pipeline- VectorDB To LLM Output Generation

In [52]:
import os
from dotenv import load_dotenv
load_dotenv()

#print(os.getenv("groq-api-key"))

True

In [37]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [50]:
class GroqLLM:
    def __init__(self, model_name: str = "llama-3.1-8b-instant", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("groq-api-key")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
    

In [51]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("groq-api-key"))
    qruery = "What is Retrieval-Augmented Generation (RAG)?"
    context = "Retrieval-Augmented Generation (RAG) is a technique that combines retrieval of relevant documents with generative language models to produce more accurate and informative responses. It allows the model to access external knowledge sources, such as a vector database of document embeddings, to enhance its understanding and generate better answers to user queries."
    response = groq_llm.generate_response(query=qruery, context=rag_retriever.retrieve(qruery)[0]['content'])
    print(f"Response: {response}")
    print("Groq LLM initialized successfully!")   
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None


Initialized Groq LLM with model: llama-3.1-8b-instant
Retrieving documents for query: 'What is Retrieval-Augmented Generation (RAG)?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 58.93it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


Response: Based on the provided context, Retrieval-Augmented Generation (RAG) is a concept discussed in the paper "Retrieval-Augmented Generation (RAG), Bus Inf Syst Eng 67(4):551–561 (2025)" by M. Klesel and H. F. Wittmann. Unfortunately, the provided snippet does not contain a detailed explanation of RAG. However, it appears to be a research topic in the field of computer science or information systems.

To provide a more accurate answer, I would recommend accessing the full paper or searching for additional resources on the topic.
Groq LLM initialized successfully!
